# STT Processing Workflow with Groq

Этот ноутбук содержит воркфлоу для обработки текста, полученного из STT (Speech-to-Text).
1. **Первый проход**: LLM (Groq) принимает расшифровку STT (Input) и разбивает текст, формируя структурированный JSON с раскадровкой ответов на вопросы.
2. **Цикл оценок**: Выполняется 6 независимых вызовов LLM для оценки каждого из ответов по критериям (q1-q6).
3. **Объединение (Aggregation)**: Все оценки алгоритмически собираются в единый финальный JSON документ.

In [ ]:
# Устанавливаем необходимые зависимости
!pip install -q groq python-dotenv

In [103]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

# Загружаем переменные окружения из .env (предполагаем наличие GROQ_API_KEY)
load_dotenv()

# Инициализируем клиента Groq
# Обязательно добавьте Ваш токен в переменную окружения $GROQ_API_KEY
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Выбираем модель (llama3-70b-8192 отлично подходит для сложных промтов)
MODEL_NAME = os.environ.get("GROQ_MODEL", "llama3-70b-8192")

def read_prompt(filename):
    """Вспомогательная функция для чтения файлов промптов."""
    path = os.path.join("prompts", filename)
    with open(path, "r", encoding="utf-8") as f:
         return f.read()
    
def read_stt_txt(filename):
    """Вспомогательная функция для чтения текстовых файлов с результатами STT."""
    with open(filename, "r", encoding="utf-8") as f:
         return f.read()

def call_groq(prompt_text, system_message="Вы полезный HR ассистент.", require_json=True, max_tokens=None, max_retries=3):
    """Делает вызов к Groq API и возвращает распарсенный JSON."""
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt_text}
    ]
    
    # Задаем response_format для гарантированного возврата JSON
    response_args = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": 0.1,
        "max_tokens": max_tokens
    }
    if require_json:
        response_args["response_format"] = {"type": "json_object"}
    
    response = client.chat.completions.create(**response_args)
    content = response.choices[0].message.content
    
    if require_json:
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            print(f"Ошибка парсинга JSON:\n{content}")
            return None
    return content

In [111]:
# Тестовый пример STT (замените на чтение из вашего реального источника/файла)
stt_input = read_stt_txt("input.txt")
# --- ШАГ 1: Первый проход LLM (Парсинг текста на раскадровку вопросов) ---
print("Запуск первого прохода LLM (Парсер текста)...")

# Мы используем главный промпт-парсер как системный промпт
parser_system_prompt = read_prompt("prompt_main_parser.txt")

# Пользовательский ввод — это только текст STT
parser_user_prompt = stt_input.strip()

parsed_stt_json = call_groq(
    prompt_text=parser_user_prompt, 
    system_message=parser_system_prompt,
    require_json=True, # Включаем обратно строгий JSON
    max_tokens=8000, # Добавляем больше токенов для длинного текста
    max_retries=3 # Добавляем повторные попытки на случай ошибок парсинга
)

print("\nРезультат первого прохода (Раскадровка STT текста):")
print(json.dumps(parsed_stt_json, indent=4, ensure_ascii=False))

Запуск первого прохода LLM (Парсер текста)...

Результат первого прохода (Раскадровка STT текста):
{
    "questions": {
        "q1_text": "why I apply to inVision U, because I feel I need this type environment, not only for study like passive, but place where people do real things, make project, discuss, try, fail, improve, and I think for me this is important now because I learn better when I am inside serious people and serious process, not when I just sit and listen, and also I think when around you people who want more from life, you also start ask more from yourself, and I want this",
        "q2_text": "about program, I think more leadership and entrepreneurship part, because I like when situation is not clear yet, maybe little chaos, and somebody need start, like okay what is problem, what first, who can do what, how we make first version, because I am not person who know everything, no, but I feel interest when from confusion you make some movement, some structure, and I want 

In [112]:
# --- ШАГ 2: Цикл 6 LLM проходов для постановки оценки каждому ответу (q1-q6) ---

evaluations = {}

if parsed_stt_json and isinstance(parsed_stt_json, dict):
    print("Запуск цикла оценок (6 независимых LLM проходов)...")
    
    # Достаем блок с ответами из распарсенного JSON (как указано в prompt_main_parser.txt)
    questions_data = parsed_stt_json.get("questions", {})
    
    # Итерируемся от 1 до 6
    for i in range(1, 7):
        q_key = f"q{i}"
        q_text_key = f"q{i}_text"
        prompt_filename = f"q{i}_prompt.txt"
        
        # Получаем ответ кандидата
        candidate_answer = questions_data.get(q_text_key, "") or ""
        print(f"[{q_key}] Оцениваем ответ: {candidate_answer[:50]}...")
        
        # Читаем локальный промпт оценки (он будет system_message)
        try:
            eval_system_prompt = read_prompt(prompt_filename)
        except FileNotFoundError:
            print(f"Промпт {prompt_filename} не найден в папке 'prompts', пропускаем...")
            continue
            
        # Получаем оценку
        eval_result = call_groq(
            prompt_text=candidate_answer, 
            system_message=eval_system_prompt,
            require_json=True
        )
        
        # Сохраняем оценку в общий словарь
        evaluations[q_key] = eval_result
        
else:
    print("Ошибка на Шаге 1: Распарсенный JSON пуст или парсинг не удался.")

print("\nОценки по вопросам успешно проставлены (или цикл завершен).")

Запуск цикла оценок (6 независимых LLM проходов)...
[q1] Оцениваем ответ: why I apply to inVision U, because I feel I need t...
[q2] Оцениваем ответ: about program, I think more leadership and entrepr...
[q3] Оцениваем ответ: major challenge for me was beginning in new academ...
[q4] Оцениваем ответ: long term I want build things which help other peo...
[q5] Оцениваем ответ: for me leadership is not loud talking or looking s...
[q6] Оцениваем ответ: yes my family support me, maybe they cannot explai...

Оценки по вопросам успешно проставлены (или цикл завершен).


In [ ]:
# --- ШАГ 3: Алгоритмическое объединение данных в единый финальный JSON --- 

def extract_score(evaluations, question_key, metric_name):
    """Вспомогательная функция для безопасного извлечения оценок из JSON LLM"""
    try:
        if question_key in evaluations and evaluations[question_key]:
            # Ищем нужную метрику в массиве "scores", если такой формат
            if "scores" in evaluations[question_key]:
                for item in evaluations[question_key]["scores"]:
                    if isinstance(item, dict) and item.get("metric_name") == metric_name:
                        return float(item.get("score", 0))
    except Exception:
        pass
    return 0.0

# 1. Извлекаем все баллы по метрикам и вопросам
scores = {
    "q1_motivation": extract_score(evaluations, "q1", "motivation"),
    "q1_planning": extract_score(evaluations, "q1", "planning"),
    
    "q2_motivation": extract_score(evaluations, "q2", "motivation"),
    "q2_planning": extract_score(evaluations, "q2", "planning"),
    
    "q3_resilience": extract_score(evaluations, "q3", "resilience"),
    "q3_leadership": extract_score(evaluations, "q3", "leadership"),
    "q3_values": extract_score(evaluations, "q3", "values"),
    
    "q4_planning": extract_score(evaluations, "q4", "planning"),
    "q4_motivation": extract_score(evaluations, "q4", "motivation"),
    
    "q5_leadership": extract_score(evaluations, "q5", "leadership"),
    "q5_values": extract_score(evaluations, "q5", "values"),
    
    "q6_social_support": extract_score(evaluations, "q6", "social_support"),
    "q6_resilience": extract_score(evaluations, "q6", "resilience"),
    "q6_motivation": extract_score(evaluations, "q6", "motivation"),
}

# 2. Считаем агрегированные метрики по формулам
Agg_M = (0.35 * scores["q1_motivation"]) + (0.20 * scores["q2_motivation"]) + (0.35 * scores["q4_motivation"]) + (0.10 * scores["q6_motivation"])
Agg_P = (0.15 * scores["q1_planning"]) + (0.35 * scores["q2_planning"]) + (0.50 * scores["q4_planning"])
Agg_R = (0.80 * scores["q3_resilience"]) + (0.20 * scores["q6_resilience"])
Agg_L = (0.30 * scores["q3_leadership"]) + (0.70 * scores["q5_leadership"])
Agg_V = (0.40 * scores["q3_values"]) + (0.60 * scores["q5_values"])
Agg_S = 1.0 * scores["q6_social_support"]

# 3. Считаем глобальные индексы
LeadershipIndex = (0.35 * Agg_L) + (0.20 * Agg_R) + (0.20 * Agg_P) + (0.15 * Agg_M) + (0.10 * Agg_V)
AdmissionsPotential = (0.25 * Agg_L) + (0.20 * Agg_P) + (0.20 * Agg_M) + (0.20 * Agg_R) + (0.10 * Agg_V) + (0.05 * Agg_S)


final_combined_output = {
    "workflow_status": "success",
    "stt_length": len(stt_input) if stt_input else 0,
    "candidate_breakdown": parsed_stt_json.get("questions", parsed_stt_json) if isinstance(parsed_stt_json, dict) else parsed_stt_json,
    "llm_evaluations": evaluations,
    "aggregated_metrics": {
        "Motivation": round(Agg_M, 2),
        "Planning": round(Agg_P, 2),
        "Resilience": round(Agg_R, 2),
        "Leadership": round(Agg_L, 2),
        "Values": round(Agg_V, 2),
        "Social_Support": round(Agg_S, 2)
    },
    "global_score": {
        "LeadershipIndex": round(LeadershipIndex, 2),
        "AdmissionsPotential": round(AdmissionsPotential, 2)
    }
}

# Формируем итоговую JSON-строку
final_json_str = json.dumps(final_combined_output, indent=4, ensure_ascii=False)

print("=== ФИНАЛЬНЫЙ СТРУКТУРИРОВАННЫЙ JSON ОБЪЕКТ ===")
print(final_json_str)

# Сохраняем в файл 
with open("final_interview_evaluation.json", "w", encoding="utf-8") as f:
    f.write(final_json_str)

print("Результат сохранен в 'final_interview_evaluation.json'")

# Тестовый пример STT (замените на чтение из вашего реального источника/файла)
stt_input = read_stt_txt("input.txt")
# --- ШАГ 1: Первый проход LLM (Парсинг текста на раскадровку вопросов) ---
print("Запуск первого прохода LLM (Парсер текста)...")

# Мы используем главный промпт-парсер как системный промпт
parser_system_prompt = read_prompt("prompt_main_parser.txt")

# Пользовательский ввод — это только текст STT
parser_user_prompt = stt_input.strip()

parsed_stt_json = call_groq(
    prompt_text=parser_user_prompt, 
    system_message=parser_system_prompt,
    require_json=True, # Включаем обратно строгий JSON
    max_tokens=8000, # Добавляем больше токенов для длинного текста
    max_retries=3 # Добавляем повторные попытки на случай ошибок парсинга
)

print("\nРезультат первого прохода (Раскадровка STT текста):")
print(json.dumps(parsed_stt_json, indent=4, ensure_ascii=False))

=== ФИНАЛЬНЫЙ СТРУКТУРИРОВАННЫЙ JSON ОБЪЕКТ ===
{
    "workflow_status": "success",
    "stt_length": 4066,
    "candidate_breakdown": {
        "q1_text": "why I apply to inVision U, because I feel I need this type environment, not only for study like passive, but place where people do real things, make project, discuss, try, fail, improve, and I think for me this is important now because I learn better when I am inside serious people and serious process, not when I just sit and listen, and also I think when around you people who want more from life, you also start ask more from yourself, and I want this",
        "q2_text": "about program, I think more leadership and entrepreneurship part, because I like when situation is not clear yet, maybe little chaos, and somebody need start, like okay what is problem, what first, who can do what, how we make first version, because I am not person who know everything, no, but I feel interest when from confusion you make some movement, some struc

In [6]:
# ...existing code...
# Указываем адрес запущенного MLflow сервера
mlflow.set_tracking_uri("http://127.0.0.1:5000") 

mlflow.set_experiment("llm_v2_comparison")
# ...existing code...

2026/04/01 23:13:59 INFO mlflow.tracking.fluent: Experiment with name 'llm_v2_comparison' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1775067239056, experiment_id='1', last_update_time=1775067239056, lifecycle_stage='active', name='llm_v2_comparison', tags={}, workspace='default'>

In [ ]:
import os
import json
import time
from datetime import datetime
from groq import Groq
from dotenv import load_dotenv

try:
    import mlflow
except ImportError:
    mlflow = None

# Загружаем переменные окружения
load_dotenv()
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
MODEL_NAME = "llama-3.1-8b-instant"  #os.environ.get("GROQ_MODEL", "llama3-70b-8192")

TESTS = [
    {"id": "good_1", "label": "good", "path": "test_inputs/good_candidate_1.txt"},
    {"id": "good_2", "label": "good", "path": "test_inputs/good_candidate_2.txt"},
    {"id": "mid_1", "label": "middle", "path": "test_inputs/middle_candidate_1.txt"},
    {"id": "mid_2", "label": "middle", "path": "test_inputs/middle_candidate_2.txt"},
    {"id": "bad_1", "label": "bad", "path": "test_inputs/bad_candidate_1.txt"},
    {"id": "bad_2", "label": "bad", "path": "test_inputs/bad_candidate_2.txt"},
]

def read_prompt(filename):
    path = os.path.join("prompts", filename)
    with open(path, "r", encoding="utf-8") as f:
         return f.read()

def read_stt_txt(filename):
    with open(filename, "r", encoding="utf-8") as f:
         return f.read()

def call_groq(prompt_text, system_message="Вы полезный HR ассистент.", require_json=True, max_tokens=None, max_retries=3):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt_text}
    ]
    response_args = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": 0.1,
        "max_tokens": max_tokens
    }
    if require_json:
        response_args["response_format"] = {"type": "json_object"}
    
    last_error = None
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(**response_args)
            content = response.choices[0].message.content
            if require_json:
                return json.loads(content)
            return content
        except Exception as e:
            last_error = e
            time.sleep(1)
            continue
    print(f"Ошибка вызова API: {last_error}")
    return None

def extract_score(evaluations, question_key, metric_name):
    try:
        if question_key in evaluations and evaluations[question_key]:
            if "scores" in evaluations[question_key]:
                for item in evaluations[question_key]["scores"]:
                    if isinstance(item, dict) and item.get("metric_name") == metric_name:
                        return float(item.get("score", 0))
    except Exception:
        pass
    return 0.0

# ...existing code...
def run_tests_and_log_mlflow():
    if mlflow is None:
        print("MLflow не установлен. Установите: pip install mlflow")
        return

    mlflow.set_experiment("llm_v2_comparison")
    run_name = f"{MODEL_NAME}__v2_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    results = []
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({"model_name": MODEL_NAME, "num_tests": len(TESTS)})

        for t in TESTS:
            print(f"Processing {t['id']} ({t['label']})...")
            start_time = time.time()

            try:
                stt_text = read_stt_txt(t["path"])
                output = process_candidate(stt_text)
                latency = time.time() - start_time

                # "total" для графиков как в примере: берем AdmissionsPotential
                total_score = float(output["globals"]["AdmissionsPotential"])

                # Условная проверка валидности JSON-ответа
                parsed_ok = isinstance(output.get("parsed_stt"), dict)
                evals = output.get("evaluations") or {}
                evals_ok = isinstance(evals, dict) and any(v is not None for v in evals.values())
                valid_json = 1.0 if (parsed_ok and evals_ok) else 0.0

                row = {
                    "test_id": t["id"],
                    "label": t["label"],
                    "latency_sec": latency,
                    "metrics": output["metrics"],
                    "globals": output["globals"],
                    "total_score": total_score,
                    "valid_json": valid_json,
                    "status": "success",
                }
            except Exception as e:
                latency = time.time() - start_time
                row = {
                    "test_id": t["id"],
                    "label": t["label"],
                    "latency_sec": latency,
                    "status": f"error: {str(e)}",
                    "valid_json": 0.0,
                }

            results.append(row)

        success_results = [r for r in results if r["status"] == "success"]
        failed_results = [r for r in results if r["status"] != "success"]

        # Базовые метрики
        mlflow.log_metric("success_rate", len(success_results) / len(TESTS))
        mlflow.log_metric("num_failed", float(len(failed_results)))

        avg_latency = sum(r["latency_sec"] for r in results) / len(results) if results else 0.0
        mlflow.log_metric("avg_latency_sec", avg_latency)

        valid_json_rate = sum(r.get("valid_json", 0.0) for r in results) / len(results) if results else 0.0
        mlflow.log_metric("valid_json_rate", valid_json_rate)

        # Средние по классам (good/middle/bad)
        for label, metric_name in [
            ("good", "good_avg_total"),
            ("middle", "middle_avg_total"),
            ("bad", "bad_avg_total"),
        ]:
            label_rows = [r for r in success_results if r.get("label") == label]
            if label_rows:
                avg_total = sum(r["total_score"] for r in label_rows) / len(label_rows)
                mlflow.log_metric(metric_name, avg_total)

        # Старые агрегаты оставляем
        if success_results:
            avg_li = sum(r["globals"]["LeadershipIndex"] for r in success_results) / len(success_results)
            avg_ap = sum(r["globals"]["AdmissionsPotential"] for r in success_results) / len(success_results)
            mlflow.log_metrics({"avg_LeadershipIndex": avg_li, "avg_AdmissionsPotential": avg_ap})

        # Артефакт с результатами
        os.makedirs("mlflow_outputs", exist_ok=True)
        out_path = f"mlflow_outputs/results_v2_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

        mlflow.log_artifact(out_path)
        print(f"✅ Тестирование завершено, результаты сохранены в MLflow и в {out_path}")
# ...existing code...


# Запуск по 6 тестам с трэкингом mlflow
run_tests_and_log_mlflow()


Processing good_1 (good)...
Processing good_2 (good)...
Processing mid_1 (middle)...
Processing mid_2 (middle)...
Processing bad_1 (bad)...


In [13]:
import requests
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'id': 'openai/gpt-oss-safeguard-20b', 'object': 'model', 'created': 1761708789, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536}, {'id': 'qwen/qwen3-32b', 'object': 'model', 'created': 1748396646, 'owned_by': 'Alibaba Cloud', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 40960}, {'id': 'canopylabs/orpheus-v1-english', 'object': 'model', 'created': 1766186316, 'owned_by': 'Canopy Labs', 'active': True, 'context_window': 4000, 'public_apps': None, 'max_completion_tokens': 50000}, {'id': 'whisper-large-v3-turbo', 'object': 'model', 'created': 1728413088, 'owned_by': 'OpenAI', 'active': True, 'context_window': 448, 'public_apps': None, 'max_completion_tokens': 448}, {'id': 'meta-llama/llama-prompt-guard-2-86m', 'object': 'model', 'created': 1748632165, 'owned_by': 'Meta', 'active': True, 'context_window': 512, 'public_apps': None, 'max_completion_tokens

In [20]:
models = [
    "llama-3.1-8b-instant",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "groq/compound",
    "groq/compound-mini",
    "allam-2-7b",
]

for model in models:
    print(f"\n===== START MODEL: {model} =====")
    MODEL_NAME = model
    try:
        run_tests_and_log_mlflow()
        print(f"===== DONE MODEL: {model} =====")
    except Exception as e:
        print(f"===== ERROR MODEL: {model} | {e} =====")



===== START MODEL: llama-3.1-8b-instant =====
Processing good_1 (good)...
Ошибка вызова API: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kbqe2mgqe8qsmwh24hdmnakt` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 7144, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Processing good_2 (good)...
Ошибка вызова API: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kbqe2mgqe8qsmwh24hdmnakt` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 7151, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Processing mid_1 (middle)...
Ошибка выз